In [ ]:
library(sceasy)
library(reticulate)
library(Seurat)
library(harmony)
library(ggplot2)

In [ ]:
## h5ad2rds

In [ ]:
h5ad_file <- "data/processed/raw_snRNA_h5ad_file"
rds_file <- "data/processed/raw_snRNA_h5ad_file"

sceasy::convertFormat(h5ad_file, from="anndata", to="seurat", main_layer="counts", outFile=rds_file)

In [ ]:
## 每个样本四批次整合

In [ ]:
seuratObj1 <- readRDS("data/processed/raw_snRNA_h5ad_file")
seuratObj2 <- readRDS("data/processed/raw_snRNA_h5ad_file")
seuratObj3 <- readRDS("data/processed/raw_snRNA_h5ad_file")
seuratObj4 <- readRDS("data/processed/raw_snRNA_h5ad_file")

In [ ]:
# 提取原始计数矩阵
counts1 <- GetAssayData(seuratObj1, assay = "RNA", slot = "counts")
counts2 <- GetAssayData(seuratObj2, assay = "RNA", slot = "counts")
counts3 <- GetAssayData(seuratObj3, assay = "RNA", slot = "counts")
counts4 <- GetAssayData(seuratObj4, assay = "RNA", slot = "counts")

# 使用原始计数矩阵创建新的 Seurat 对象并进行质控
data1 <- CreateSeuratObject(counts = counts1, project = "B2PFC-1", min.cells = 3, min.features = 200)
data2 <- CreateSeuratObject(counts = counts2, project = "B2PFC-2", min.cells = 3, min.features = 200)
data3 <- CreateSeuratObject(counts = counts3, project = "B2PFC-3", min.cells = 3, min.features = 200)
data4 <- CreateSeuratObject(counts = counts4, project = "B2PFC-4", min.cells = 3, min.features = 200)

In [ ]:
# 合并数据
combined <- merge(data1, y = c(data2, data3, data4))

# 检查每个 Seurat 对象的细胞数量
cat("Cells in data1:", ncol(data1), "\n")
cat("Cells in data2:", ncol(data2), "\n")
cat("Cells in data3:", ncol(data3), "\n")
cat("Cells in data4:", ncol(data4), "\n")

# 创建批次标签向量
batch_vector <- rep(1:4, times = c(ncol(data1), ncol(data2), ncol(data3), ncol(data4)))

In [ ]:
# 绘制细胞的特征数（基因数）分布
VlnPlot(combined, features = "nFeature_RNA", pt.size = 0.1)
# 绘制细胞的 UMI 数量分布
VlnPlot(combined, features = "nCount_RNA", pt.size = 0.1)

In [ ]:
## 重新QC nFeature_RNA>1000 & percent.mt<5

In [ ]:
combined[["percent.mt"]] <- PercentageFeatureSet(combined, pattern = "^MT-")
combined <- subset(combined, subset = nFeature_RNA > 1000 & percent.mt < 5)
combined

In [ ]:
combined <- NormalizeData(combined, normalization.method = "LogNormalize", scale.factor = 10000)
combined <- FindVariableFeatures(combined, selection.method = "vst", nfeatures = 3000)
combined <- ScaleData(combined)
combined <- RunPCA(combined, features = VariableFeatures(object = combined))

In [ ]:
## 去批次之后聚类分群

In [ ]:
combined <- RunHarmony(combined, group.by.vars = "batch", plot_convergence = FALSE)
combined <- FindNeighbors(combined, reduction = "harmony", dims = 1:30)
combined <- FindClusters(combined, resolution = 0.2)
combined <- RunUMAP(combined, reduction = "harmony", dims = 1:30)

In [ ]:
## 可视化检查

In [ ]:
DimPlot(combined, reduction = "umap", group.by = "batch", label = TRUE)
DimPlot(combined, reduction = "umap", group.by = "seurat_clusters", label = TRUE)

In [ ]:
## 计算差异基因

In [ ]:
Idents(combined) <- "seurat_clusters"
markers <- FindAllMarkers(combined, only.pos = TRUE, min.pct = 0.25, logfc.threshold = 0.25)
write.csv(markers, file = "data/processed/cortex_analysis_file", row.names = FALSE)

In [ ]:
## 过滤差异基因

In [ ]:
# 检查是否存在pct.1和pct.2列
if("pct.1" %in% colnames(markers) && "pct.2" %in% colnames(markers)) {
  # 添加ratio列
  markers$ratio <- (markers$pct.1 - markers$pct.2) / markers$pct.1
} else {
  stop("pct.1 或 pct.2 列不存在，请检查数据！")
}

markers_filtered <- markers[markers$ratio > 0.5, ]